# Generalized Knowledge Distillation Trainer
Link: 
- https://huggingface.co/docs/trl/en/gkd_trainer

The key aspects of GKD are:

- It addresses the train-inference distribution mismatch in auto-regressive sequence models by training the student model on its self-generated output sequences.
- GKD allows flexibility in choosing different divergence measures between student and teacher models via the generalized Jensen-Shannon Divergence (JSD), which can be useful when the student lacks the capacity to fully mimic the teacher.

## Qwen2

In [2]:
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.experimental.gkd import GKDConfig, GKDTrainer

In [4]:
SEED = 67
N_TRAIN = 2000
N_VAL = 200

In [21]:
ds = load_dataset(
    "csebuetnlp/xlsum",
    data_dir="ukrainian",
    revision="refs/convert/parquet",
)

train_raw = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
val_raw = ds["validation"].shuffle(seed=SEED).select(range(N_VAL))
print(len(train_raw), len(val_raw))

2000 200


In [24]:
def xlsum_prompt(record, max_chars=3000):
    title = re.sub(r" {2,}", " ", record["title"])
    text = re.sub(r" {2,}", " ", record["text"])[:max_chars]
    return (
        "Підсумуй цю статтю:\n "
        f"Назва: {title}\n "
        # f"Посилання: {record['url']}\n"
        f"Текст статті: \n{text}\n "
    )
def to_messages(record):
    return {
        "messages": [
            {"role": "user", "content": xlsum_prompt(record)},
            {"role": "assistant", "content": record["summary"]},
        ]
    }

train_dataset = train_raw.map(to_messages, remove_columns=train_raw.column_names)
eval_dataset = val_raw.map(to_messages, remove_columns=val_raw.column_names)
train_dataset[0]["messages"][0]["content"][:400]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map: 100%|██████████| 200/200 [00:00<00:00, 11229.13 examples/s]


'Підсумуй цю статтю:\n Назва: О.Мороз закликає протистояти шахрайству влади\n Текст статті: \nУ листі до лідера "Нашої України" та провідника комуністів, Олександр Мороз звертає увагу на прояви того, що він називає "масштабним чорним піаром" вже на початковому етапі передвиборчої кампанії. Олександр Мороз не виключає, що це може перетворити виборчу кампанію - цитата - "на один з найбрудніших періодів '

In [10]:
student_id = "Qwen/Qwen2-0.5B-Instruct"
teacher_id = "Qwen/Qwen2-1.5B-Instruct"

use_cuda = torch.cuda.is_available()
bf16 = use_cuda and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16 else (torch.float16 if use_cuda else torch.float32)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(student_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# The model to optimise
model = AutoModelForCausalLM.from_pretrained(student_id, dtype=dtype)
# The teacher model to calculate the KL divergence against
teacher_model = AutoModelForCausalLM.from_pretrained(teacher_id, dtype=dtype)

teacher_model.save_pretrained("../models/qwen2-1.5b-teacher")
tokenizer.save_pretrained("../models/qwen2-1.5b-teacher")

Loading weights: 100%|██████████| 338/338 [00:08<00:00, 39.55it/s]


In [ ]:
training_args = GKDConfig(
    output_dir="../models/gkd-qwen2-0.5b-xlsum",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_length=1024,
    lmbda=1.0,
    beta=0.5,
    temperature=0.9,
    max_new_tokens=64,
    seq_kd=False,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    bf16=bf16,
    fp16=use_cuda and not bf16,
    seed=SEED,
    report_to="none",
)

trainer = GKDTrainer(
    model=model,
    teacher_model=teacher_model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
trainer.train()
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/Users/ija/Desktop/small-LLM/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


KeyboardInterrupt: 

In [ ]:
print(trainer.evaluate()["eval_loss"]) # Jensen–Shannon

### Rouge_L score

In [13]:
from rouge_score import rouge_scorer

ModuleNotFoundError: No module named 'rouge_score'

In [ ]:
N_TEST = 100
test_raw = ds["test"].shuffle(seed=SEED).select(range(N_TEST))
records = list(test_raw)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [ ]:
def generate_summary(model, record, max_new_tokens=64, max_prompt_tokens=960):
    messages = [{"role": "user", "content": xlsum_prompt(record)}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False) #Qwen-format prompt
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False) #tokenized prompts
    if inputs["input_ids"].shape[1] > max_prompt_tokens:
        inputs = {key: value[:, -max_prompt_tokens:] for key, value in inputs.items()} # shorten tokenized prompt 
    inputs = {key: value.to(model.device) for key, value in inputs.items()} # put tensors on the same dvice
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # no greedy choice
            pad_token_id=tokenizer.pad_token_id, 
            eos_token_id=model.generation_config.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def rouge_on_xlsum(model, records):
    totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    model.eval()
    for record in records:
        prediction = generate_summary(model, record)
        scores = scorer.score(record["summary"], prediction)
        for key in totals:
            totals[key] += scores[key].fmeasure
    return {key: value / len(records) for key, value in totals.items()}

def prepare(model):
    return model.to(device)

In [ ]:
rouge_student = rouge_on_xlsum(prepare(model), records)
print("distilled", rouge_student)
model.to("cpu")

rouge_teacher = rouge_on_xlsum(prepare(teacher_model), records)
print("teacher", rouge_teacher)
teacher_model.to("cpu")

In [ ]:
base = AutoModelForCausalLM.from_pretrained(student_id, dtype=dtype)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("base", rouge_on_xlsum(prepare(base), records))
base.to("cpu")

In [ ]:
print("gold:", records[0]["summary"])
print("distilled:", generate_summary(prepare(model), records[0]))

In [ ]:
for key in rouge_teacher:
    print(key, rouge_student[key] - rouge_teacher[key])

In [ ]:
quality = rouge_student / rouge_teacher
print(quality)

In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters())

student_n = param_count(model)
teacher_n = param_count(teacher_model)

print(f"student: {student_n / 1e6:.1f} M")
print(f"teacher: {teacher_n / 1e6:.1f} M")
print(f"compression: {teacher_n / student_n:.2f}x")

In [ ]:
def weight_mb(model):
    nbytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return nbytes / 1024**2

print(f"student: {weight_mb(model):.0f} MB")
print(f"teacher: {weight_mb(teacher_model):.0f} MB")